# Polynomial Regression — Hands-on Programming

**Goal.** Build a `PolynomialFeatures` transformer from scratch (univariate and multivariate), compose it with the `LinearRegressionOLS` class from the previous folder, run k-fold cross-validation to pick a degree, and verify everything against scikit-learn's `PolynomialFeatures` + `LinearRegression`.

**Role of this notebook.** Implementation and empirical validation. Every formula used here was derived in a previous notebook:

| Used here | Where derived |
|---|---|
| $\Phi_d(x)$ = (1, x, x^2, …, x^d) | `02_mathematics.ipynb` (1.1) |
| Multivariate $\Phi_{d,q}$ | `02_mathematics.ipynb` (4.1) |
| Closed-form OLS on $\Phi$ | `02_mathematics.ipynb` (2.2) |
| Feature scaling before $\Phi$ | `03_optimization.ipynb` §3 |
| Orthogonal basis | `03_optimization.ipynb` §4 |
| k-fold CV | `04_statistics.ipynb` §3.1 |

**Prerequisites.** All four prior notebooks in this folder, plus `01_linear_regression/05_hands_on_programming.ipynb` (the `LinearRegressionOLS` class).

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → `04_statistics` → **`05_hands_on_programming`**.

**Plan.**

1. Imports, seeding.
2. Implement `PolynomialFeatures` from scratch — univariate and multivariate total-degree.
3. Sanity-check against `sklearn.preprocessing.PolynomialFeatures`.
4. Glue it to `LinearRegressionOLS` (a lightweight in-file copy of the class from the previous folder) → a `PolynomialRegressor` pipeline.
5. Cross-check the full pipeline against `sklearn` on synthetic 1-D data.
6. k-fold CV: pick d on a noisy sine.
7. Multivariate sanity demo on the diabetes dataset — feature count vs. degree.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random
from itertools import combinations_with_replacement
from math import comb

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures as SKPolyFeats
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. Polynomial feature transformer

The multivariate total-degree feature map (eq. 4.1 of `02_mathematics.ipynb`) enumerates all monomials x^$\alpha$ with |$\alpha$| $\le$ d. The standard programming trick to enumerate those monomials is `itertools.combinations_with_replacement` over the feature indices, repeated for each degree from 0 to d. We build the same enumeration scikit-learn does.

In [ ]:
class PolynomialFeatures:
    """Total-degree polynomial features (eq. 4.1 of 02_mathematics).

    Matches sklearn.preprocessing.PolynomialFeatures(degree=d, include_bias=True).
    For a single input feature this collapses to (1, x, x^2, ..., x^d) (eq. 1.1).
    """

    def __init__(self, degree=2, include_bias=True):
        self.degree       = degree
        self.include_bias = include_bias

    def _powers(self, n_input):
        """Yield all multi-indices alpha with |alpha| <= degree, in sklearn's order."""
        start_deg = 0 if self.include_bias else 1
        for deg in range(start_deg, self.degree + 1):
            for combo in combinations_with_replacement(range(n_input), deg):
                # combo is a tuple of feature indices; build alpha as a tuple of counts.
                alpha = [0] * n_input
                for idx in combo:
                    alpha[idx] += 1
                yield tuple(alpha)

    def fit(self, X):
        X = np.atleast_2d(X)
        if X.shape[0] == 1 and X.ndim == 2 and X.shape[1] != 1:
            # caller passed a single row; keep as is
            pass
        self.n_input_features_ = X.shape[1]
        self.powers_ = list(self._powers(self.n_input_features_))
        return self

    def transform(self, X):
        X = np.atleast_2d(X)
        n, q = X.shape
        out = np.empty((n, len(self.powers_)))
        for j, alpha in enumerate(self.powers_):
            col = np.ones(n)
            for k, a_k in enumerate(alpha):
                if a_k:
                    col *= X[:, k] ** a_k
            out[:, j] = col
        return out

    def fit_transform(self, X):
        return self.fit(X).transform(X)

# Tiny demo: q = 2, d = 2 -> 6 features (1, x1, x2, x1^2, x1 x2, x2^2)
pf = PolynomialFeatures(degree=2)
X_small = np.array([[2.0, 3.0]])
Z = pf.fit_transform(X_small)
print("input X       =", X_small)
print("powers (alpha) =", pf.powers_)
print("features Φ(X)  =", Z)

### 1.1 Sanity-check against scikit-learn

On a random small dataset, our `PolynomialFeatures` should produce the same matrix as `sklearn.preprocessing.PolynomialFeatures` (modulo column ordering — they happen to use the same ordering).

In [ ]:
X_demo = rng.normal(size=(7, 3))
for d in [1, 2, 3, 4]:
    ours = PolynomialFeatures(degree=d).fit_transform(X_demo)
    skl  = SKPolyFeats(degree=d, include_bias=True).fit_transform(X_demo)
    print(f"d = {d}:  shape ours = {ours.shape},  shape sklearn = {skl.shape},  "
          f"max abs diff = {np.max(np.abs(ours - skl)):.2e}")

**Reading.** Identical to machine precision. The combinatorial count C(d + q, q) of `02_mathematics.ipynb` (4.2) is visible in the column count: at q = 3 and d = 4, both produce 35 columns = C(7, 3).

## 2. The full pipeline: PolynomialRegressor

We compose `PolynomialFeatures` with a *lightweight* OLS estimator. The class below is a self-contained copy of `LinearRegressionOLS` from `01_linear_regression/05_hands_on_programming.ipynb`, kept here so this notebook runs without imports from sibling folders.

In [ ]:
class LinearRegressionOLS:
    """Same as 01_linear_regression/05_hands_on_programming.ipynb.

    Adds a bias column internally; lstsq for numerical stability (02_mathematics §4.3).
    """
    def fit(self, X, y):
        n = X.shape[0]
        X_design = np.column_stack([np.ones(n), X])
        theta, *_ = np.linalg.lstsq(X_design, y, rcond=None)
        self.theta_     = theta
        self.intercept_ = float(theta[0])
        self.coef_      = theta[1:]
        return self

    def predict(self, X):
        return self.intercept_ + X @ self.coef_


class PolynomialRegressor:
    """Pipeline: PolynomialFeatures(degree=d, include_bias=False) -> LinearRegressionOLS.

    include_bias=False inside PolynomialFeatures because LinearRegressionOLS adds its own
    intercept column — having two of them gives a redundant column and a singular design.
    """
    def __init__(self, degree=2):
        self.degree = degree

    def fit(self, X, y):
        self.poly_ = PolynomialFeatures(degree=self.degree, include_bias=False).fit(X)
        Z = self.poly_.transform(X)
        self.ols_ = LinearRegressionOLS().fit(Z, y)
        return self

    def predict(self, X):
        return self.ols_.predict(self.poly_.transform(X))

# Quick smoke test on a curve.
x = np.linspace(-2, 2, 60).reshape(-1, 1)
y = np.sin(2 * x).ravel() + rng.normal(0, 0.1, x.shape[0])
model = PolynomialRegressor(degree=5).fit(x, y)
y_pred = model.predict(x)
print(f"Training MSE on sin(2x) with d = 5: {np.mean((y - y_pred) ** 2):.5f}")

### 2.1 Cross-check against sklearn's pipeline

Predictions of `PolynomialRegressor(degree=d)` and `Pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression())` should match to machine precision on the same data — they solve the same OLS problem with the same lstsq backend.

In [ ]:
for d in [1, 3, 5, 8]:
    ours = PolynomialRegressor(degree=d).fit(x, y)
    skl = Pipeline([
        ("poly", SKPolyFeats(degree=d, include_bias=False)),
        ("ols",  LinearRegression()),
    ]).fit(x, y)
    pred_ours = ours.predict(x)
    pred_skl  = skl.predict(x)
    print(f"d = {d}:  max |ours - sklearn pipeline| = {np.max(np.abs(pred_ours - pred_skl)):.2e}")

## 3. Pick d by 5-fold cross-validation

Repeating §3 of `04_statistics.ipynb` with the actual pipeline. Sweep d, run 5-fold CV, plot CV MSE vs. d, pick the minimum.

In [ ]:
# Reuse the noisy-sine data from §2.
degrees = list(range(1, 16))
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

cv_mse = []
train_mse = []
for d in degrees:
    fold_mses = []
    for tr_idx, te_idx in kf.split(x):
        m = PolynomialRegressor(degree=d).fit(x[tr_idx], y[tr_idx])
        fold_mses.append(mean_squared_error(y[te_idx], m.predict(x[te_idx])))
    cv_mse.append(np.mean(fold_mses))
    # In-sample MSE on the full data, for comparison.
    m_full = PolynomialRegressor(degree=d).fit(x, y)
    train_mse.append(mean_squared_error(y, m_full.predict(x)))

d_best = degrees[int(np.argmin(cv_mse))]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(degrees, train_mse, "o-", color="steelblue", label="train MSE")
ax.plot(degrees, cv_mse,    "o-", color="crimson",   label="5-fold CV MSE")
ax.axvline(d_best, color="crimson", ls=":", label=f"CV-best d = {d_best}")
ax.set_yscale("log")
ax.set_xlabel("polynomial degree d")
ax.set_ylabel("MSE (log)")
ax.set_title("CV picks the bottom of the U on a noisy sine")
ax.legend()
plt.show()

print(f"Train MSE keeps going down (overfit), CV MSE U-shaped, CV-best d = {d_best}")

### 3.1 Visualise the selected fit

Plot the data, the true function, and three fits: underfit (d = 1), CV-best (d = d_best), and overfit (d = 14).

In [ ]:
xs = np.linspace(-2.2, 2.2, 400).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4), sharey=True)
for ax, (d, label) in zip(axes, [(1, "underfit"), (d_best, "CV-best"), (14, "overfit")]):
    m = PolynomialRegressor(degree=d).fit(x, y)
    ax.plot(xs, np.sin(2 * xs), color="black", ls="--", lw=1, label="true f")
    ax.scatter(x, y, alpha=0.6, edgecolor="k")
    ax.plot(xs, m.predict(xs), color="crimson", lw=2, label=f"degree {d}")
    ax.set_xlabel("x")
    ax.set_title(f"{label}  (d = {d})")
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

## 4. Multivariate sanity demo — feature count blows up

Take the 10-feature diabetes regression dataset and look at how many columns the polynomial design has at d = 1, 2, 3, 4. This is the curse of dimensionality from `02_mathematics.ipynb` §4.2 made concrete.

Then fit `PolynomialRegressor(d)` for each d and report train / test RMSE. With only 442 patients, anything past d = 2 has so many parameters that it overfits hard — and CV would prefer a low degree.

In [ ]:
data = load_diabetes()
X_full, y_full = data.data, data.target
n_full, q = X_full.shape
print(f"diabetes: n = {n_full}, q = {q}")
print()
print(f"{'d':>3}  {'# features':>11}  {'C(d+q, q)':>11}")
for d in [1, 2, 3, 4]:
    Z = PolynomialFeatures(degree=d).fit_transform(X_full)
    print(f"{d:>3}  {Z.shape[1]:>11}  {comb(d + q, q):>11}")

In [ ]:
# Train/test split + sweep d, plot the bias-variance curve at scale.
perm = rng.permutation(n_full)
n_train = int(0.8 * n_full)
tr, te = perm[:n_train], perm[n_train:]
X_tr, X_te = X_full[tr], X_full[te]
y_tr, y_te = y_full[tr], y_full[te]

rmse_tr, rmse_te, n_feats = [], [], []
for d in [1, 2, 3]:
    model = PolynomialRegressor(degree=d).fit(X_tr, y_tr)
    rmse_tr.append(float(np.sqrt(mean_squared_error(y_tr, model.predict(X_tr)))))
    rmse_te.append(float(np.sqrt(mean_squared_error(y_te, model.predict(X_te)))))
    n_feats.append(model.poly_.transform(X_tr[:1]).shape[1])

print(f"{'d':>3}  {'# features':>11}  {'RMSE_train':>12}  {'RMSE_test':>12}")
for d, p_, rtr, rte in zip([1, 2, 3], n_feats, rmse_tr, rmse_te):
    print(f"{d:>3}  {p_:>11}  {rtr:>12.2f}  {rte:>12.2f}")

**Reading.** d = 1 is plain linear regression — same numbers as the previous folder. d = 2 has 65 features (intercept + 10 linear + 55 quadratic / interaction) and already begins to overfit: train RMSE drops, test RMSE rises. d = 3 has 285 features on 353 training points — the design is barely full-rank and the model memorises noise. This is exactly the curse-of-dimensionality warning of `02_mathematics.ipynb` §4.2. In practice this is why people reach for:

- **Regularised** regression (Ridge → `03_ridge_regression/`, Lasso → `04_lasso_regression/`) to keep the design large but tame the variance.
- **Interaction-only** polynomial features (drop pure powers x_k^2) when the interesting non-linearity is in feature *interactions* but not in marginal curvature.
- **Tree-based** models (`06_decision_tree`, `07_random_forest`, `08_gradient_boosting`) when the non-linearity is high-dimensional and unstructured.

## Takeaway

- **PolynomialFeatures is just an enumeration.**   All total-degree monomials, enumerated by `combinations_with_replacement`. Our implementation matches `sklearn.preprocessing.PolynomialFeatures` to machine precision.
- **The pipeline is just OLS on $\Phi$(X).**   `PolynomialRegressor(d)` = `PolynomialFeatures(d)` → `LinearRegressionOLS`. No new fitting algorithm. Sklearn's `Pipeline(PolynomialFeatures, LinearRegression)` gives the same predictions to machine precision.
- **k-fold CV picks d.**   On the noisy sine, 5-fold CV reproduces the bias–variance U-shape and selects a degree near the oracle (`04_statistics.ipynb` §2.1).
- **Multivariate degree is dangerous.**   Diabetes (q = 10): d = 2 → 65 features, d = 3 → 285, d = 4 → 1 001. With only ~350 training points, anything past d = 2 overfits — exactly what `02_mathematics.ipynb` §4.2 predicts.

**This concludes the Polynomial Regression folder.** Next: `03_ridge_regression/` keeps $\Phi_d$ but adds an L^2 penalty to control coefficient magnitudes — the variance side of the bias–variance trade-off becomes a continuous tuning knob, rather than the discrete choice of d.